In [5]:
import os
import pandas as pd
import numpy as np
from PIL import Image
import torch
import torchvision.transforms as transforms
import torchvision.models as models
from collections import defaultdict

In [6]:
class UnifiedImageDatasetBuilder:
    def __init__(self, model_type='resnet50'):
        """
        Build unified dataframe with embeddings and metadata
        """
        self.model_type = model_type
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"Using device: {self.device}")
        
        # Initialize model
        if model_type == 'resnet50':
            self.model = models.resnet50(pretrained=True)
            self.model = torch.nn.Sequential(*list(self.model.children())[:-1])
            self.embedding_dim = 2048
            
        elif model_type == 'resnet18':
            self.model = models.resnet18(pretrained=True)
            self.model = torch.nn.Sequential(*list(self.model.children())[:-1])
            self.embedding_dim = 512
            
        elif model_type == 'vgg16':
            self.model = models.vgg16(pretrained=True)
            self.model.classifier = torch.nn.Sequential(*list(self.model.classifier.children())[:-1])
            self.embedding_dim = 4096
            
        self.model.to(self.device)
        self.model.eval()
        
        # Preprocessing
        self.preprocess = transforms.Compose([
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                               std=[0.229, 0.224, 0.225])
        ])
        
        print(f"Model: {model_type}, Embedding dimension: {self.embedding_dim}")
    
    def extract_embedding(self, image_path):
        """Extract embedding from image"""
        image = Image.open(image_path).convert('RGB')
        image_tensor = self.preprocess(image).unsqueeze(0).to(self.device)
        
        with torch.no_grad():
            embedding = self.model(image_tensor)
            embedding = embedding.cpu().numpy().flatten()
        
        return embedding
    
    def get_image_metadata(self, image_path):
        """Extract basic image metadata"""
        image = Image.open(image_path)
        width, height = image.size
        mode = image.mode
        
        return {
            'image_width': width,
            'image_height': height,
            'image_mode': mode,
            'aspect_ratio': round(width / height, 3)
        }
    
    def build_unified_dataframe(self, anonymous_path, countries_path):
        """
        Build unified dataframe with all information
        """
        unified_data = []
        image_id = 0
        
        print("Building unified dataframe...")
        
        # Process Anonymous Dataset
        print("Processing anonymous images...")
        anonymous_files = [f for f in os.listdir(anonymous_path) if f.endswith('.jpg')]
        
        for i, filename in enumerate(sorted(anonymous_files)):
            image_path = os.path.join(anonymous_path, filename)
            
            # Extract embedding
            embedding = self.extract_embedding(image_path)
            
            # Get image metadata
            metadata = self.get_image_metadata(image_path)
            
            # Create unified row
            row = {
                'image_id': image_id,
                'filename': filename,
                'original_filename': filename,
                'dataset_source': 'anonymous',
                'label': 'ANONYMOUS',
                'country_code': 'ANONYMOUS',
                'is_labeled': False,
                'image_index_in_category': i,
                **metadata  # Add image metadata
            }
            
            # Add embedding features
            for j, emb_val in enumerate(embedding):
                row[f'feature_{j}'] = emb_val
            
            unified_data.append(row)
            image_id += 1
            
            if (i + 1) % 50 == 0:
                print(f"  Processed {i + 1}/{len(anonymous_files)} anonymous images")
        
        print(f"Completed {len(anonymous_files)} anonymous images")
        
        # Process Countries Dataset
        print("Processing country-labeled images...")
        country_files = [f for f in os.listdir(countries_path) if f.endswith('.jpg')]
        
        # Group by country
        country_groups = defaultdict(list)
        for filename in country_files:
            country_code = filename.split('-')[0]
            country_groups[country_code].append(filename)
        
        total_country_images = 0
        for country_code, files in country_groups.items():
            print(f"Processing {country_code}: {len(files)} images")
            
            for i, filename in enumerate(sorted(files)):
                image_path = os.path.join(countries_path, filename)
                
                # Extract embedding
                embedding = self.extract_embedding(image_path)
                
                # Get image metadata
                metadata = self.get_image_metadata(image_path)
                
                # Extract image number from filename (e.g., "AT-5.jpg" -> 5)
                try:
                    image_number = int(filename.split('-')[1].split('.')[0])
                except:
                    image_number = i + 1
                
                # Create unified row
                row = {
                    'image_id': image_id,
                    'filename': filename,
                    'original_filename': filename,
                    'dataset_source': 'countries',
                    'label': country_code,
                    'country_code': country_code,
                    'is_labeled': True,
                    'image_index_in_category': i,
                    'original_image_number': image_number,
                    **metadata  # Add image metadata
                }
                
                # Add embedding features
                for j, emb_val in enumerate(embedding):
                    row[f'feature_{j}'] = emb_val
                
                unified_data.append(row)
                image_id += 1
                total_country_images += 1
                
                if total_country_images % 50 == 0:
                    print(f"  Processed {total_country_images} country images")
        
        # Create unified DataFrame
        print("Creating unified DataFrame...")
        unified_df = pd.DataFrame(unified_data)
        
        # Add some computed columns
        unified_df['total_pixels'] = unified_df['image_width'] * unified_df['image_height']
        unified_df['is_square'] = unified_df['aspect_ratio'] == 1.0
        unified_df['is_landscape'] = unified_df['aspect_ratio'] > 1.0
        unified_df['is_portrait'] = unified_df['aspect_ratio'] < 1.0
        
        return unified_df

def create_unified_dataset(anonymous_path, countries_path, output_path, model_type='resnet50'):
    """
    Main function to create unified dataset
    """
    
    # Build unified dataframe
    builder = UnifiedImageDatasetBuilder(model_type)
    unified_df = builder.build_unified_dataframe(anonymous_path, countries_path)
    
    # Save to CSV
    unified_df.to_csv(output_path, index=False)
    
    # Print comprehensive summary
    print("\n" + "="*70)
    print("UNIFIED DATASET CREATED!")
    print("="*70)
    
    print(f"Total images: {len(unified_df)}")
    print(f"Total features: {builder.embedding_dim}")
    print(f"DataFrame shape: {unified_df.shape}")
    
    print("\nDataset Distribution:")
    label_counts = unified_df['label'].value_counts()
    for label, count in label_counts.items():
        percentage = (count / len(unified_df)) * 100
        print(f"  {label}: {count} images ({percentage:.1f}%)")
    
    print("\nDataFrame Columns:")
    print("Metadata columns:")
    metadata_cols = [col for col in unified_df.columns if not col.startswith('feature_')]
    for col in metadata_cols:
        print(f"  - {col}")
    
    print(f"\nFeature columns: feature_0 to feature_{builder.embedding_dim-1}")
    
    print(f"\nImage Statistics:")
    print(f"  Average width: {unified_df['image_width'].mean():.1f} pixels")
    print(f"  Average height: {unified_df['image_height'].mean():.1f} pixels")
    print(f"  Average aspect ratio: {unified_df['aspect_ratio'].mean():.3f}")
    print(f"  Square images: {unified_df['is_square'].sum()}")
    print(f"  Landscape images: {unified_df['is_landscape'].sum()}")
    print(f"  Portrait images: {unified_df['is_portrait'].sum()}")
    
    print(f"\nSaved to: {output_path}")
    
    return unified_df

# YOUR PATHS
anonymous_path = r'C:\Saim_Files\TU GRAZ\Semester 2\Visual Analytics\Postcrossing-Dataset\anonymous'
countries_path = r'C:\Saim_Files\TU GRAZ\Semester 2\Visual Analytics\Postcrossing-Dataset\countries'
output_path = r'C:\Saim_Files\TU GRAZ\Semester 2\Visual Analytics\Postcrossing-Dataset\unified_dataset.csv'

# Create unified dataset
MODEL_TYPE = 'resnet50'  # Choose: 'resnet50', 'resnet18', 'vgg16'

print("Creating unified image dataset...")
unified_df = create_unified_dataset(anonymous_path, countries_path, output_path, MODEL_TYPE)

print(f"\n✅ SUCCESS: Unified dataset created!")
print(f"\nFirst few rows preview:")
print(unified_df[['image_id', 'filename', 'label', 'country_code', 'image_width', 'image_height']].head())

print(f"\nDataFrame info:")
print(f"Memory usage: {unified_df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
print(f"Ready for analysis!")

Creating unified image dataset...
Using device: cpu


C:\Users\saima\AppData\Local\Programs\Python\Python311\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\saima\AppData\Local\Programs\Python\Python311\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Model: resnet50, Embedding dimension: 2048
Building unified dataframe...
Processing anonymous images...
  Processed 50/909 anonymous images
  Processed 100/909 anonymous images
  Processed 150/909 anonymous images
  Processed 200/909 anonymous images
  Processed 250/909 anonymous images
  Processed 300/909 anonymous images
  Processed 350/909 anonymous images
  Processed 400/909 anonymous images
  Processed 450/909 anonymous images
  Processed 500/909 anonymous images
  Processed 550/909 anonymous images
  Processed 600/909 anonymous images
  Processed 650/909 anonymous images
  Processed 700/909 anonymous images
  Processed 750/909 anonymous images
  Processed 800/909 anonymous images
  Processed 850/909 anonymous images
  Processed 900/909 anonymous images
Completed 909 anonymous images
Processing country-labeled images...
Processing AT: 16 images
Processing AU: 18 images
Processing BE: 20 images
  Processed 50 country images
Processing BG: 2 images
Processing BH: 1 images
Processing

In [24]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score
import cv2
import os
from PIL import Image
import warnings
warnings.filterwarnings('ignore')



In [25]:
class CompletePostcardAnalysisSystem:
    def __init__(self, csv_path, image_base_path):
        """
        Complete analysis system for postcard dataset
        """
        self.csv_path = csv_path
        self.image_base_path = image_base_path
        self.df = None
        self.feature_matrix = None
        self.color_features = None
        self.color_df = None
        self.topic_results = {}
        self.color_results = {}
        self.pca_results = {}
        self.tsne_results = {}
        self.recommendation_df = None
        self.reports = {}
        
        print("🚀 Initializing Complete Postcard Analysis System...")
        self.load_data()
    
    def load_data(self):
        """Load and prepare the unified dataset"""
        print("📂 Loading unified dataset...")
        self.df = pd.read_csv(self.csv_path)
        
        # Extract feature columns (embeddings)
        feature_cols = [col for col in self.df.columns if col.startswith('feature_')]
        self.feature_matrix = self.df[feature_cols].values
        
        print(f"✅ Dataset loaded: {self.df.shape}")
        print(f"✅ Feature dimensions: {self.feature_matrix.shape}")
        
        # Handle missing values
        if np.isnan(self.feature_matrix).any():
            print("🔧 Handling missing values...")
            self.feature_matrix = np.nan_to_num(self.feature_matrix)
        
        return self.df, self.feature_matrix
    


In [26]:
    def extract_color_features(self, max_images=2305):
        """Extract color features from images"""
        sample_size = min(max_images, len(self.df))
        print(f"🎨 Extracting color features from {sample_size} images...")
        
        # Sample images for processing
        if sample_size < len(self.df):
            sample_indices = np.random.choice(len(self.df), sample_size, replace=False)
            sample_df = self.df.iloc[sample_indices].copy()
        else:
            sample_indices = range(len(self.df))
            sample_df = self.df.copy()
        
        color_features = []
        color_records = []
        processed_count = 0
        
        for idx, (orig_idx, row) in enumerate(sample_df.iterrows()):
            try:
                filename = row['filename']
                
                # Try different image paths
                possible_paths = [
                    os.path.join(self.image_base_path, 'anonymous', filename),
                    os.path.join(self.image_base_path, 'countries', filename),
                    os.path.join(self.image_base_path, filename),
                    os.path.join(self.image_base_path, 'merged_labeled_dataset', filename)
                ]
                
                image_path = None
                for path in possible_paths:
                    if os.path.exists(path):
                        image_path = path
                        break
                
                if image_path and os.path.exists(image_path):
                    # Load and analyze image
                    image = cv2.imread(image_path)
                    if image is not None:
                        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
                        color_feat = self._analyze_image_colors(image_rgb)
                        color_features.append(color_feat)
                        
                        # Store record
                        color_record = {
                            'original_index': orig_idx,
                            'filename': filename,
                            'dataset_source': row.get('dataset_source', 'unknown'),
                            'label': row.get('label', 'unknown'),
                            'country_code': row.get('country_code', 'unknown'),
                            'avg_red': color_feat[0],
                            'avg_green': color_feat[1],
                            'avg_blue': color_feat[2],
                            'brightness': color_feat[3],
                            'saturation': color_feat[4],
                            'red_tendency': color_feat[5],
                            'blue_tendency': color_feat[6],
                            'processed_successfully': True
                        }
                        color_records.append(color_record)
                    else:
                        # Default values for failed images
                        color_features.append([0.5, 0.5, 0.5, 0.5, 0.5, 0.33, 0.33])
                        color_records.append(self._create_default_color_record(orig_idx, row, "load_failed"))
                else:
                    # Default values for missing images
                    color_features.append([0.5, 0.5, 0.5, 0.5, 0.5, 0.33, 0.33])
                    color_records.append(self._create_default_color_record(orig_idx, row, "not_found"))
                
                processed_count += 1
                if processed_count % 200 == 0:
                    print(f"  📊 Processed {processed_count}/{sample_size} images...")
                    
            except Exception as e:
                color_features.append([0.5, 0.5, 0.5, 0.5, 0.5, 0.33, 0.33])
                color_records.append(self._create_default_color_record(orig_idx, row, f"error: {str(e)[:30]}"))
        
        self.color_features = np.array(color_features)
        self.color_df = pd.DataFrame(color_records)
        
        success_count = sum(self.color_df['processed_successfully'])
        print(f"✅ Color extraction complete: {success_count}/{sample_size} successful")
        
        return self.color_features, self.color_df


In [28]:
    
    def _analyze_image_colors(self, image):
        """Analyze color properties of an image"""
        # Resize for faster processing
        image_small = cv2.resize(image, (100, 100))
        hsv = cv2.cvtColor(image_small, cv2.COLOR_RGB2HSV)
        
        # RGB averages
        avg_r = np.mean(image_small[:,:,0]) / 255.0
        avg_g = np.mean(image_small[:,:,1]) / 255.0
        avg_b = np.mean(image_small[:,:,2]) / 255.0
        
        # Brightness and saturation
        brightness = (avg_r + avg_g + avg_b) / 3.0
        avg_saturation = np.mean(hsv[:,:,1]) / 255.0
        
        # Color tendencies
        total = avg_r + avg_g + avg_b + 1e-6
        red_tendency = avg_r / total
        blue_tendency = avg_b / total
        
        return [avg_r, avg_g, avg_b, brightness, avg_saturation, red_tendency, blue_tendency]


In [29]:
    
    def _create_default_color_record(self, orig_idx, row, reason):
        """Create default record for failed color extraction"""
        return {
            'original_index': orig_idx,
            'filename': row.get('filename', 'unknown'),
            'dataset_source': row.get('dataset_source', 'unknown'),
            'label': row.get('label', 'unknown'),
            'country_code': row.get('country_code', 'unknown'),
            'avg_red': 0.5, 'avg_green': 0.5, 'avg_blue': 0.5,
            'brightness': 0.5, 'saturation': 0.5,
            'red_tendency': 0.33, 'blue_tendency': 0.33,
            'processed_successfully': False,
            'failure_reason': reason
        }
    


In [30]:
    def find_optimal_k_topics(self, max_k=15):
        """Find optimal K for topic clustering"""
        print("🔍 Finding optimal K for topic clustering...")
        
        # Sample for faster computation
        sample_size = min(2305, len(self.feature_matrix))
        if len(self.feature_matrix) > sample_size:
            indices = np.random.choice(len(self.feature_matrix), sample_size, replace=False)
            X_sample = self.feature_matrix[indices]
        else:
            X_sample = self.feature_matrix
        
        # Standardize
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X_sample)
        
        # Test different K values
        k_range = range(2, max_k + 1)
        inertias = []
        silhouette_scores = []
        
        for k in k_range:
            kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
            labels = kmeans.fit_predict(X_scaled)
            
            inertias.append(kmeans.inertia_)
            sil_score = silhouette_score(X_scaled, labels)
            silhouette_scores.append(sil_score)
            
            if k % 3 == 0:
                print(f"  K={k}: silhouette={sil_score:.3f}")
        
        # Find optimal K
        best_k = k_range[np.argmax(silhouette_scores)]
        best_score = max(silhouette_scores)
        
        # Store results
        self.k_analysis = {
            'k_range': k_range,
            'inertias': inertias,
            'silhouette_scores': silhouette_scores,
            'optimal_k': best_k,
            'best_score': best_score
        }
        
        print(f"✅ Optimal K for topics: {best_k} (silhouette: {best_score:.3f})")
        return best_k, best_score
    


In [31]:
    def perform_topic_clustering(self, k=None):
        """Perform topic clustering"""
        if k is None:
            k, _ = self.find_optimal_k_topics()
        
        print(f"🎯 Performing topic clustering with K={k}...")
        
        # Standardize features
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(self.feature_matrix)
        
        # Perform clustering
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        topic_labels = kmeans.fit_predict(X_scaled)
        
        # Add to dataframe
        self.df['topic_cluster'] = topic_labels
        
        # Calculate quality metrics
        sil_score = silhouette_score(X_scaled, topic_labels)
        
        # Store results
        self.topic_results = {
            'kmeans': kmeans,
            'scaler': scaler,
            'labels': topic_labels,
            'silhouette_score': sil_score,
            'n_clusters': k
        }
        
        print(f"✅ Topic clustering complete: {k} clusters, silhouette={sil_score:.3f}")
        return topic_labels, sil_score
    


In [32]:
    def perform_color_clustering(self, k=6):
        """Perform color clustering"""
        if self.color_features is None:
            print("🎨 Extracting color features first...")
            self.extract_color_features(max_images=2305)
        
        print(f"🌈 Performing color clustering with K={k}...")
        
        # Standardize color features
        scaler = StandardScaler()
        color_scaled = scaler.fit_transform(self.color_features)
        
        # Perform clustering
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        color_labels = kmeans.fit_predict(color_scaled)
        
        # Add to color dataframe
        self.color_df['color_cluster'] = color_labels
        
        # Define color names
        color_names = {
            0: "Warm_Tones",
            1: "Cool_Tones", 
            2: "Neutral_Colors",
            3: "Vibrant_Bright",
            4: "Dark_Moody",
            5: "Balanced_Natural"
        }
        
        self.color_df['color_name'] = self.color_df['color_cluster'].map(
            lambda x: color_names.get(x, f"Color_{x}")
        )
        
        # Store results
        self.color_results = {
            'kmeans': kmeans,
            'scaler': scaler,
            'labels': color_labels,
            'n_clusters': k,
            'color_names': color_names
        }
        
        print(f"✅ Color clustering complete: {k} clusters")
        return color_labels, self.color_df
    


In [33]:
    def perform_pca_analysis(self):
        """Perform PCA analysis on features"""
        print("📊 Performing PCA analysis...")
        
        # Standardize features
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(self.feature_matrix)
        
        # PCA with different components
        components_list = [2, 10, 50, 100]
        
        for n_comp in components_list:
            if n_comp <= min(X_scaled.shape):
                pca = PCA(n_components=n_comp, random_state=42)
                X_pca = pca.fit_transform(X_scaled)
                
                self.pca_results[n_comp] = {
                    'pca': pca,
                    'transformed': X_pca,
                    'explained_variance_ratio': pca.explained_variance_ratio_,
                    'cumulative_variance': np.cumsum(pca.explained_variance_ratio_),
                    'total_variance_explained': pca.explained_variance_ratio_.sum()
                }
                
                print(f"  PCA-{n_comp}: {pca.explained_variance_ratio_.sum():.3f} variance explained")
        
        print("✅ PCA analysis complete")
        return self.pca_results


In [34]:
    
    def perform_tsne_analysis(self):
        """Perform t-SNE analysis"""
        print("📈 Performing t-SNE analysis...")
        
        # Sample for t-SNE (it's slow on large datasets)
        sample_size = min(2305, len(self.feature_matrix))
        if len(self.feature_matrix) > sample_size:
            indices = np.random.choice(len(self.feature_matrix), sample_size, replace=False)
            X_sample = self.feature_matrix[indices]
            df_sample = self.df.iloc[indices]
        else:
            X_sample = self.feature_matrix
            df_sample = self.df
            indices = range(len(self.df))
        
        # Standardize
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X_sample)
        
        # t-SNE
        tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, sample_size//4))
        X_tsne = tsne.fit_transform(X_scaled)
        
        self.tsne_results = {
            'tsne': tsne,
            'transformed': X_tsne,
            'sample_indices': indices,
            'sample_df': df_sample
        }
        
        print(f"✅ t-SNE complete on {sample_size} samples")
        return self.tsne_results


In [35]:
    
    def analyze_all_clusters(self):
        """Comprehensive analysis of all clusters"""
        print("\n" + "="*80)
        print("📊 COMPREHENSIVE CLUSTER ANALYSIS")
        print("="*80)
        
        # Topic cluster analysis
        if 'topic_cluster' in self.df.columns:
            self._analyze_topic_clusters()
        
        # Color cluster analysis
        if hasattr(self, 'color_df') and self.color_df is not None and 'color_cluster' in self.color_df.columns:
            self._analyze_color_clusters()
    


In [36]:
    def _analyze_topic_clusters(self):
        """Analyze topic clusters in detail"""
        print("\n🎯 TOPIC CLUSTER DETAILED ANALYSIS")
        print("-" * 50)
        
        for cluster_id in sorted(self.df['topic_cluster'].unique()):
            cluster_data = self.df[self.df['topic_cluster'] == cluster_id]
            
            print(f"\n📂 Topic Cluster {cluster_id}")
            print("=" * 30)
            print(f"Size: {len(cluster_data)} images ({len(cluster_data)/len(self.df)*100:.1f}%)")
            
            # Dataset source distribution
            if 'dataset_source' in cluster_data.columns:
                source_dist = cluster_data['dataset_source'].value_counts()
                print(f"Sources: {dict(source_dist)}")
            
            # Country distribution
            if 'country_code' in cluster_data.columns:
                country_dist = cluster_data['country_code'].value_counts().head(5)
                print(f"Top countries: {dict(country_dist)}")
            
            # Image characteristics
            if 'aspect_ratio' in cluster_data.columns:
                avg_aspect = cluster_data['aspect_ratio'].mean()
                print(f"Avg aspect ratio: {avg_aspect:.2f}")
                
                orientation = "Landscape" if avg_aspect > 1.2 else "Portrait" if avg_aspect < 0.8 else "Square"
                print(f"Dominant orientation: {orientation}")
            
            # Sample files
            samples = cluster_data['filename'].head(5).tolist()
            print(f"Samples: {samples}")

    


In [37]:
    
    def _analyze_color_clusters(self):
        """Analyze color clusters in detail"""
        print("\n🎨 COLOR CLUSTER DETAILED ANALYSIS")
        print("-" * 50)
        
        for cluster_id in sorted(self.color_df['color_cluster'].unique()):
            cluster_data = self.color_df[self.color_df['color_cluster'] == cluster_id]
            
            print(f"\n🌈 Color Cluster {cluster_id}: {cluster_data['color_name'].iloc[0]}")
            print("=" * 40)
            print(f"Size: {len(cluster_data)} images ({len(cluster_data)/len(self.color_df)*100:.1f}%)")
            
            # Color characteristics
            print(f"Avg RGB: ({cluster_data['avg_red'].mean():.2f}, {cluster_data['avg_green'].mean():.2f}, {cluster_data['avg_blue'].mean():.2f})")
            print(f"Avg brightness: {cluster_data['brightness'].mean():.2f}")
            print(f"Avg saturation: {cluster_data['saturation'].mean():.2f}")
            
            # Country distribution
            if 'country_code' in cluster_data.columns:
                country_dist = cluster_data['country_code'].value_counts().head(3)
                print(f"Top countries: {dict(country_dist)}")
            
            # Sample files
            samples = cluster_data['filename'].head(3).tolist()
            print(f"Samples: {samples}")
    


In [38]:
    def create_all_visualizations(self):
        """Create comprehensive visualizations"""
        print("\n📈 Creating comprehensive visualizations...")
        
        # Create figure with subplots
        fig = plt.figure(figsize=(20, 16))
        
        # 1. Topic cluster distribution
        ax1 = plt.subplot(3, 4, 1)
        if 'topic_cluster' in self.df.columns:
            cluster_counts = self.df['topic_cluster'].value_counts().sort_index()
            bars = ax1.bar(cluster_counts.index, cluster_counts.values, color='lightblue', alpha=0.7)
            ax1.set_title('Topic Cluster Sizes')
            ax1.set_xlabel('Cluster ID')
            ax1.set_ylabel('Number of Images')
            ax1.grid(True, alpha=0.3)
            
            # Add value labels
            for bar in bars:
                height = bar.get_height()
                ax1.text(bar.get_x() + bar.get_width()/2., height + 5,
                        f'{int(height)}', ha='center', va='bottom', fontsize=8)
        
        # 2. PCA variance explained
        ax2 = plt.subplot(3, 4, 2)
        if 2 in self.pca_results:
            pca_50 = self.pca_results.get(50, self.pca_results[2])
            cumvar = pca_50['cumulative_variance'][:20]  # First 20 components
            ax2.plot(range(1, len(cumvar)+1), cumvar, 'bo-')
            ax2.set_title('PCA Cumulative Variance')
            ax2.set_xlabel('Component')
            ax2.set_ylabel('Cumulative Variance Explained')
            ax2.grid(True, alpha=0.3)
        
        # 3. PCA 2D visualization (topic clusters)
        ax3 = plt.subplot(3, 4, 3)
        if 2 in self.pca_results and 'topic_cluster' in self.df.columns:
            X_pca = self.pca_results[2]['transformed']
            scatter = ax3.scatter(X_pca[:, 0], X_pca[:, 1], 
                                c=self.df['topic_cluster'], cmap='tab10', alpha=0.6, s=20)
            ax3.set_title('PCA: Topic Clusters')
            ax3.set_xlabel('PC1')
            ax3.set_ylabel('PC2')
            ax3.grid(True, alpha=0.3)
        
        # 4. t-SNE visualization (topic clusters)
        ax4 = plt.subplot(3, 4, 4)
        if self.tsne_results and 'topic_cluster' in self.tsne_results['sample_df'].columns:
            X_tsne = self.tsne_results['transformed']
            sample_clusters = self.tsne_results['sample_df']['topic_cluster']
            ax4.scatter(X_tsne[:, 0], X_tsne[:, 1], 
                       c=sample_clusters, cmap='tab10', alpha=0.6, s=20)
            ax4.set_title('t-SNE: Topic Clusters')
            ax4.set_xlabel('t-SNE 1')
            ax4.set_ylabel('t-SNE 2')
            ax4.grid(True, alpha=0.3)
        
        # 5. Color cluster distribution
        ax5 = plt.subplot(3, 4, 5)
        if hasattr(self, 'color_df') and self.color_df is not None and 'color_cluster' in self.color_df.columns:
            color_counts = self.color_df['color_cluster'].value_counts().sort_index()
            bars = ax5.bar(color_counts.index, color_counts.values, color='lightgreen', alpha=0.7)
            ax5.set_title('Color Cluster Sizes')
            ax5.set_xlabel('Color Cluster ID')
            ax5.set_ylabel('Number of Images')
            ax5.grid(True, alpha=0.3)
        
        # 6. Dataset source by topic cluster
        ax6 = plt.subplot(3, 4, 6)
        if 'topic_cluster' in self.df.columns and 'dataset_source' in self.df.columns:
            source_cluster = pd.crosstab(self.df['topic_cluster'], self.df['dataset_source'])
            source_cluster.plot(kind='bar', stacked=True, ax=ax6)
            ax6.set_title('Dataset Source by Topic Cluster')
            ax6.set_xlabel('Topic Cluster')
            ax6.set_ylabel('Count')
            ax6.legend(title='Source')
            ax6.tick_params(axis='x', rotation=0)
        
        # 7. Country distribution (top countries)
        ax7 = plt.subplot(3, 4, 7)
        if 'country_code' in self.df.columns:
            top_countries = self.df['country_code'].value_counts().head(10)
            bars = ax7.bar(range(len(top_countries)), top_countries.values, color='orange', alpha=0.7)
            ax7.set_title('Top 10 Countries')
            ax7.set_xlabel('Country')
            ax7.set_ylabel('Number of Images')
            ax7.set_xticks(range(len(top_countries)))
            ax7.set_xticklabels(top_countries.index, rotation=45)
        
        # 8. Aspect ratio distribution
        ax8 = plt.subplot(3, 4, 8)
        if 'aspect_ratio' in self.df.columns:
            ax8.hist(self.df['aspect_ratio'], bins=30, alpha=0.7, color='purple')
            ax8.set_title('Aspect Ratio Distribution')
            ax8.set_xlabel('Aspect Ratio')
            ax8.set_ylabel('Frequency')
            ax8.axvline(x=1.0, color='red', linestyle='--', label='Square')
            ax8.legend()
        
        # 9. Color cluster visualization (if available)
        ax9 = plt.subplot(3, 4, 9)
        if hasattr(self, 'color_df') and self.color_df is not None and 'color_cluster' in self.color_df.columns:
            # Create color feature space visualization
            ax9.scatter(self.color_df['brightness'], self.color_df['saturation'],
                       c=self.color_df['color_cluster'], cmap='viridis', alpha=0.6)
            ax9.set_title('Color Space: Brightness vs Saturation')
            ax9.set_xlabel('Brightness')
            ax9.set_ylabel('Saturation')
        
        # 10. PCA component importance
        ax10 = plt.subplot(3, 4, 10)
        if 2 in self.pca_results:
            pca_result = self.pca_results.get(50, self.pca_results[2])
            explained_var = pca_result['explained_variance_ratio'][:20]
            ax10.bar(range(1, len(explained_var)+1), explained_var, alpha=0.7)
            ax10.set_title('PCA Component Importance')
            ax10.set_xlabel('Component')
            ax10.set_ylabel('Variance Explained')
        
        # 11. Color name distribution
        ax11 = plt.subplot(3, 4, 11)
        if hasattr(self, 'color_df') and self.color_df is not None and 'color_name' in self.color_df.columns:
            color_name_counts = self.color_df['color_name'].value_counts()
            bars = ax11.bar(range(len(color_name_counts)), color_name_counts.values, 
                           color=['red', 'blue', 'gray', 'yellow', 'black', 'green'][:len(color_name_counts)], alpha=0.7)
            ax11.set_title('Color Cluster Names')
            ax11.set_ylabel('Count')
            ax11.set_xticks(range(len(color_name_counts)))
            ax11.set_xticklabels(color_name_counts.index, rotation=45)
        
        # 12. Summary statistics
        ax12 = plt.subplot(3, 4, 12)
        ax12.axis('off')
        
        # Create summary text
        summary_text = f"""DATASET SUMMARY

Total Images: {len(self.df):,}
Feature Dimensions: {self.feature_matrix.shape[1]:,}
        """
        
        if 'topic_cluster' in self.df.columns:
            n_topic_clusters = self.df['topic_cluster'].nunique()
            topic_sil = self.topic_results.get('silhouette_score', 0)
            summary_text += f"\nTopic Clusters: {n_topic_clusters}\nTopic Silhouette: {topic_sil:.3f}"
        
        if hasattr(self, 'color_df') and self.color_df is not None and 'color_cluster' in self.color_df.columns:
            n_color_clusters = self.color_df['color_cluster'].nunique()
            summary_text += f"\nColor Clusters: {n_color_clusters}\nColor Samples: {len(self.color_df):,}"
        
        if 2 in self.pca_results:
            pca_var = self.pca_results[2]['total_variance_explained']
            summary_text += f"\nPCA Variance (2D): {pca_var:.3f}"
        
        ax12.text(0.1, 0.9, summary_text, transform=ax12.transAxes, fontsize=10,
                 verticalalignment='top', bbox=dict(boxstyle='round', facecolor='lightgray', alpha=0.8))
        
        plt.tight_layout()
        plt.show()
        
        print("✅ All visualizations created!")
    


In [39]:
    def build_comprehensive_recommendation_system(self):
        """Build comprehensive recommendation system using all clustering results"""
        print("\n🎯 Building comprehensive recommendation system...")
        
        recommendation_data = []
        
        for idx, row in self.df.iterrows():
            # Get topic cluster info
            topic_cluster = row.get('topic_cluster', -1)
            
            # Find similar images by topic
            topic_similar = []
            if topic_cluster != -1:
                same_topic = self.df[
                    (self.df['topic_cluster'] == topic_cluster) & 
                    (self.df.index != idx)
                ]
                topic_similar = same_topic['filename'].head(15).tolist()
            
            # Get color cluster info (if available)
            color_cluster = -1
            color_similar = []
            if hasattr(self, 'color_df') and self.color_df is not None:
                color_match = self.color_df[self.color_df['filename'] == row['filename']]
                if len(color_match) > 0:
                    color_cluster = color_match['color_cluster'].iloc[0]
                    same_color = self.color_df[
                        (self.color_df['color_cluster'] == color_cluster) & 
                        (self.color_df['filename'] != row['filename'])
                    ]
                    color_similar = same_color['filename'].head(10).tolist()
            
            # Find country-based similarities
            country_similar = []
            if 'country_code' in row and row['country_code'] != 'ANONYMOUS':
                same_country = self.df[
                    (self.df['country_code'] == row['country_code']) & 
                    (self.df.index != idx)
                ]
                country_similar = same_country['filename'].head(8).tolist()
            
            # Create comprehensive recommendation record
            rec_record = {
                'image_id': idx,
                'filename': row['filename'],
                'dataset_source': row.get('dataset_source', 'unknown'),
                'label': row.get('label', 'unknown'),
                'country_code': row.get('country_code', 'unknown'),
                
                # Topic clustering info
                'topic_cluster': topic_cluster,
                'topic_similar_images': str(topic_similar),
                'topic_similarity_count': len(topic_similar),
                
                # Color clustering info
                'color_cluster': color_cluster,
                'color_similar_images': str(color_similar),
                'color_similarity_count': len(color_similar),
                
                # Country-based similarities
                'country_similar_images': str(country_similar),
                'country_similarity_count': len(country_similar),
                
                # Combined recommendation score
                'total_similar_images': len(set(topic_similar + color_similar + country_similar)),
                'recommendation_quality': len(topic_similar) + len(color_similar) * 0.7 + len(country_similar) * 0.3
            }
            
            recommendation_data.append(rec_record)
        
        self.recommendation_df = pd.DataFrame(recommendation_data)
        
        print(f"✅ Recommendation system built for {len(self.recommendation_df)} images")
        print(f"   Average topic similarities: {self.recommendation_df['topic_similarity_count'].mean():.1f}")
        print(f"   Average color similarities: {self.recommendation_df['color_similarity_count'].mean():.1f}")
        print(f"   Average country similarities: {self.recommendation_df['country_similarity_count'].mean():.1f}")
        
        return self.recommendation_df
    


In [40]:
    def generate_detailed_reports(self):
        """Generate detailed analysis reports"""
        print("\n📋 Generating detailed reports...")
        
        reports = {}
        
        # Topic cluster report
        if 'topic_cluster' in self.df.columns:
            topic_report = []
            for cluster_id in sorted(self.df['topic_cluster'].unique()):
                cluster_data = self.df[self.df['topic_cluster'] == cluster_id]
                
                report_entry = {
                    'cluster_id': cluster_id,
                    'cluster_size': len(cluster_data),
                    'percentage': len(cluster_data) / len(self.df) * 100,
                    'top_countries': list(cluster_data['country_code'].value_counts().head(5).index),
                    'anonymous_count': len(cluster_data[cluster_data['dataset_source'] == 'anonymous']),
                    'countries_count': len(cluster_data[cluster_data['dataset_source'] == 'countries']),
                    'avg_aspect_ratio': cluster_data['aspect_ratio'].mean() if 'aspect_ratio' in cluster_data.columns else None,
                    'sample_files': list(cluster_data['filename'].head(10))
                }
                topic_report.append(report_entry)
            
            reports['topic_clusters'] = pd.DataFrame(topic_report)
        
        # Color cluster report
        if hasattr(self, 'color_df') and self.color_df is not None and 'color_cluster' in self.color_df.columns:
            color_report = []
            for cluster_id in sorted(self.color_df['color_cluster'].unique()):
                cluster_data = self.color_df[self.color_df['color_cluster'] == cluster_id]
                
                report_entry = {
                    'color_cluster_id': cluster_id,
                    'color_name': cluster_data['color_name'].iloc[0] if 'color_name' in cluster_data.columns else f'Color_{cluster_id}',
                    'cluster_size': len(cluster_data),
                    'percentage': len(cluster_data) / len(self.color_df) * 100,
                    'avg_red': cluster_data['avg_red'].mean(),
                    'avg_green': cluster_data['avg_green'].mean(),
                    'avg_blue': cluster_data['avg_blue'].mean(),
                    'avg_brightness': cluster_data['brightness'].mean(),
                    'avg_saturation': cluster_data['saturation'].mean(),
                    'top_countries': list(cluster_data['country_code'].value_counts().head(3).index),
                    'sample_files': list(cluster_data['filename'].head(5))
                }
                color_report.append(report_entry)
            
            reports['color_clusters'] = pd.DataFrame(color_report)
        
        # PCA report
        if self.pca_results:
            pca_report = []
            for n_comp, result in self.pca_results.items():
                pca_entry = {
                    'n_components': n_comp,
                    'total_variance_explained': result['total_variance_explained'],
                    'first_component_variance': result['explained_variance_ratio'][0],
                    'second_component_variance': result['explained_variance_ratio'][1] if len(result['explained_variance_ratio']) > 1 else None
                }
                pca_report.append(pca_entry)
            
            reports['pca_analysis'] = pd.DataFrame(pca_report)
        
        self.reports = reports
        print(f"✅ Generated {len(reports)} detailed reports")
        return reports


In [41]:
    
    def save_all_results(self, output_dir=None):
        """Save all results to files"""
        if output_dir is None:
            output_dir = os.path.dirname(self.csv_path)
        
        print(f"\n💾 Saving all results to {output_dir}...")
        
        saved_files = {}
        
        # 1. Main dataset with topic clusters
        if 'topic_cluster' in self.df.columns:
            topic_file = os.path.join(output_dir, 'complete_dataset_with_topic_clusters.csv')
            self.df.to_csv(topic_file, index=False)
            saved_files['topic_clustered_dataset'] = topic_file
            print(f"✅ Topic clustered dataset: {os.path.basename(topic_file)}")
        
        # 2. Color clustering data
        if hasattr(self, 'color_df') and self.color_df is not None:
            color_file = os.path.join(output_dir, 'color_clusters_complete.csv')
            self.color_df.to_csv(color_file, index=False)
            saved_files['color_clusters'] = color_file
            print(f"✅ Color clusters: {os.path.basename(color_file)}")
        
        # 3. Comprehensive recommendations
        if hasattr(self, 'recommendation_df') and self.recommendation_df is not None:
            rec_file = os.path.join(output_dir, 'comprehensive_recommendations.csv')
            self.recommendation_df.to_csv(rec_file, index=False)
            saved_files['recommendations'] = rec_file
            print(f"✅ Recommendations: {os.path.basename(rec_file)}")
        
        # 4. PCA results
        if self.pca_results:
            pca_file = os.path.join(output_dir, 'pca_analysis_results.csv')
            pca_data = []
            for n_comp, result in self.pca_results.items():
                pca_data.append({
                    'n_components': n_comp,
                    'total_variance_explained': result['total_variance_explained'],
                    'explained_variance_ratios': str(result['explained_variance_ratio'].tolist())
                })
            pd.DataFrame(pca_data).to_csv(pca_file, index=False)
            saved_files['pca_results'] = pca_file
            print(f"✅ PCA results: {os.path.basename(pca_file)}")
        
        # 5. Detailed reports
        if hasattr(self, 'reports') and self.reports:
            for report_name, report_df in self.reports.items():
                report_file = os.path.join(output_dir, f'{report_name}_detailed_report.csv')
                report_df.to_csv(report_file, index=False)
                saved_files[f'{report_name}_report'] = report_file
                print(f"✅ {report_name} report: {os.path.basename(report_file)}")
        
        # 6. Analysis summary
        summary_file = os.path.join(output_dir, 'analysis_summary.txt')
        with open(summary_file, 'w') as f:
            f.write("COMPLETE POSTCARD ANALYSIS SUMMARY\n")
            f.write("=" * 50 + "\n\n")
            
            f.write(f"Dataset Information:\n")
            f.write(f"- Total images: {len(self.df):,}\n")
            f.write(f"- Feature dimensions: {self.feature_matrix.shape[1]:,}\n\n")
            
            if 'topic_cluster' in self.df.columns:
                n_topic = self.df['topic_cluster'].nunique()
                topic_sil = self.topic_results.get('silhouette_score', 0)
                f.write(f"Topic Clustering:\n")
                f.write(f"- Number of clusters: {n_topic}\n")
                f.write(f"- Silhouette score: {topic_sil:.3f}\n\n")
            
            if hasattr(self, 'color_df') and self.color_df is not None:
                n_color = self.color_df['color_cluster'].nunique()
                f.write(f"Color Clustering:\n")
                f.write(f"- Number of clusters: {n_color}\n")
                f.write(f"- Images analyzed: {len(self.color_df):,}\n\n")
            
            if self.pca_results and 2 in self.pca_results:
                pca_var = self.pca_results[2]['total_variance_explained']
                f.write(f"PCA Analysis:\n")
                f.write(f"- 2D variance explained: {pca_var:.3f}\n")
                if 50 in self.pca_results:
                    pca_50_var = self.pca_results[50]['total_variance_explained']
                    f.write(f"- 50D variance explained: {pca_50_var:.3f}\n")
                f.write("\n")
            
            f.write(f"Files Generated:\n")
            for file_type, file_path in saved_files.items():
                f.write(f"- {file_type}: {os.path.basename(file_path)}\n")
        
        saved_files['summary'] = summary_file
        print(f"✅ Analysis summary: {os.path.basename(summary_file)}")
        
        print(f"\n🎉 All results saved! {len(saved_files)} files generated.")
        return saved_files

In [42]:
    def run_complete_analysis(self, topic_k=10, color_k=6, max_color_images=2305):
        """Run the complete analysis pipeline"""
        print("\n" + "="*80)
        print("🚀 STARTING COMPLETE POSTCARD ANALYSIS PIPELINE")
        print("="*80)
        
        try:
            # Step 1: Topic clustering
            print(f"\n📊 Step 1: Topic Clustering (K={topic_k})")
            self.perform_topic_clustering(k=topic_k)
            
            # Step 2: Color feature extraction and clustering
            print(f"\n🎨 Step 2: Color Analysis (K={color_k})")
            self.extract_color_features(max_images=max_color_images)
            self.perform_color_clustering(k=color_k)
            
            # Step 3: PCA analysis
            print(f"\n📊 Step 3: PCA Analysis")
            self.perform_pca_analysis()
            
            # Step 4: t-SNE analysis
            print(f"\n📈 Step 4: t-SNE Analysis")
            self.perform_tsne_analysis()
            
            # Step 5: Comprehensive cluster analysis
            print(f"\n🔍 Step 5: Cluster Analysis")
            self.analyze_all_clusters()
            
            # Step 6: Build recommendation system
            print(f"\n🎯 Step 6: Recommendation System")
            self.build_comprehensive_recommendation_system()
            
            # Step 7: Generate reports
            print(f"\n📋 Step 7: Generate Reports")
            self.generate_detailed_reports()
            
            # Step 8: Create visualizations
            print(f"\n📈 Step 8: Create Visualizations")
            self.create_all_visualizations()
            
            # Step 9: Save all results
            print(f"\n💾 Step 9: Save Results")
            saved_files = self.save_all_results()
            
            # Final summary
            print("\n" + "="*80)
            print("✅ COMPLETE ANALYSIS FINISHED!")
            print("="*80)
            
            print(f"📊 Results Summary:")
            print(f"   • Total images analyzed: {len(self.df):,}")
            print(f"   • Topic clusters: {self.df['topic_cluster'].nunique()}")
            
            if hasattr(self, 'color_df') and self.color_df is not None:
                print(f"   • Color clusters: {self.color_df['color_cluster'].nunique()}")
            
            print(f"   • Files generated: {len(saved_files)}")
            
            if hasattr(self, 'topic_results'):
                print(f"   • Topic clustering quality: {self.topic_results['silhouette_score']:.3f}")
            
            if 2 in self.pca_results:
                print(f"   • PCA variance (2D): {self.pca_results[2]['total_variance_explained']:.3f}")
            
            print(f"\n🎯 Recommendation System Ready!")
            if hasattr(self, 'recommendation_df') and self.recommendation_df is not None:
                print(f"   • Average recommendations per image: {self.recommendation_df['total_similar_images'].mean():.1f}")
            
            print(f"\n📁 All results saved to: {os.path.dirname(self.csv_path)}")
            
            return {
                'topic_results': getattr(self, 'topic_results', None),
                'color_results': getattr(self, 'color_results', None),
                'pca_results': self.pca_results,
                'tsne_results': self.tsne_results,
                'recommendation_df': getattr(self, 'recommendation_df', None),
                'reports': getattr(self, 'reports', None),
                'saved_files': saved_files
            }
            
        except Exception as e:
            print(f"\n❌ Error during analysis: {str(e)}")
            print("Analysis stopped. Check the error above.")
            return None


In [43]:

# MAIN EXECUTION
def main():
    """Main function to run complete postcard analysis"""
    
    # YOUR PATHS - MODIFY THESE TO MATCH YOUR SETUP
    csv_path = r'C:\Saim_Files\TU GRAZ\Semester 2\Visual Analytics\Postcrossing-Dataset\unified_dataset.csv'
    image_base_path = r'C:\Saim_Files\TU GRAZ\Semester 2\Visual Analytics\Postcrossing-Dataset'
    
    print("🎨 COMPLETE POSTCARD ANALYSIS & RECOMMENDATION SYSTEM")
    print("=" * 60)
    
    try:
        # Initialize the complete system
        analysis_system = CompletePostcardAnalysisSystem(csv_path, image_base_path)
        
        # Run complete analysis
        results = analysis_system.run_complete_analysis(
            topic_k=10,           # Number of topic clusters
            color_k=6,            # Number of color clusters  
            max_color_images=2305 # Limit color analysis for speed
        )
        
        if results:
            print("\n🎉 ANALYSIS COMPLETE! Check the generated files for detailed results.")
            print("\n📁 Generated Files:")
            for file_type, file_path in results['saved_files'].items():
                print(f"   • {file_type}: {os.path.basename(file_path)}")
        else:
            print("\n❌ Analysis failed. Please check the error messages above.")
        
        return analysis_system, results
        
    except FileNotFoundError as e:
        print(f"\n❌ File not found: {e}")
        print("Please check that your CSV file exists at the specified path.")
        return None, None
        
    except Exception as e:
        print(f"\n❌ Unexpected error: {e}")
        print("Please check your file paths and data format.")
        return None, None

if __name__ == "__main__":
    system, results = main()

🎨 COMPLETE POSTCARD ANALYSIS & RECOMMENDATION SYSTEM
🚀 Initializing Complete Postcard Analysis System...
📂 Loading unified dataset...
✅ Dataset loaded: (2305, 2065)
✅ Feature dimensions: (2305, 2048)

❌ Unexpected error: 'CompletePostcardAnalysisSystem' object has no attribute 'run_complete_analysis'
Please check your file paths and data format.
